## Notebook for using the CDS API to get ERA5 data

#### Run the pip install to get the latest cdsapi

In [ ]:
#!pip install cdsapi
# created .cdsapirc file in ~/ with UID and API-key per https://pypi.org/project/cdsapi/

In [1]:
import cdsapi
import requests

### This block is for grabbing a single year, entire globe <br><br> This actually errors out and says the request is too large for netcdf, apparently

In [2]:
c = cdsapi.Client()

c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'variable': [
            #'2m_temperature', 
            'land_sea_mask',
        ],
        'year': '2023',
        'month': [
            #'01', '02', '03',
            #'04', '05', '06',
            #'07', '08', 
            '09',
            #'10', '11', '12',
        ],
        'day': [
            '01', #'02', '03',
            # '04', '05', '06',
            # '07', '08', '09',
            # '10', '11', '12',
            # '13', '14', '15',
            # '16', '17', '18',
            # '19', '20', '21',
            # '22', '23', '24',
            # '25', '26', '27',
            # '28', '29', '30',
            # '31',
        ],
        'time': [
            # '00:00', '01:00', '02:00',
            # '03:00', '04:00', '05:00',
            # '06:00', '07:00', '08:00',
            # '09:00', '10:00', '11:00',
             '12:00', #'13:00', '14:00',
            # '15:00', '16:00', '17:00',
            # '18:00', '19:00', '20:00',
            # '21:00', '22:00', '23:00',
        ],
        'format': 'netcdf',
    },
    'ERA5-2023-09-01-Full-LSM-download.nc')

2024-07-04 15:47:39,562 INFO Welcome to the CDS
2024-07-04 15:47:39,562 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/resources/reanalysis-era5-single-levels
2024-07-04 15:47:39,841 INFO Request is queued
2024-07-04 15:47:42,704 INFO Request is completed
2024-07-04 15:47:42,705 INFO Downloading https://download-0018.copernicus-climate.eu/cache-compute-0018/cache/data5/adaptor.mars.internal-1720133261.933593-1638-18-449eb970-0a03-4b82-b836-fe806558723e.nc to ERA5-2023-09-01-Full-LSM-download.nc (2M)
2024-07-04 15:47:44,709 INFO Download rate 1016.7K/s                            


Result(content_length=2086240,content_type=application/x-netcdf,location=https://download-0018.copernicus-climate.eu/cache-compute-0018/cache/data5/adaptor.mars.internal-1720133261.933593-1638-18-449eb970-0a03-4b82-b836-fe806558723e.nc)

### This block is for specific years and months, specific area lat/lon

In [ ]:
# try multiple years, same region, JJA
c = cdsapi.Client()

c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'variable': [
            '2m_temperature', 'land_sea_mask',
        ],
        'year': ['2020','2021','2022','2023'],
        'month': [
            '06', '07', '08',
        ],
        'day': [
            '01', '02', '03',
            '04', '05', '06',
            '07', '08', '09',
            '10', '11', '12',
            '13', '14', '15',
            '16', '17', '18',
            '19', '20', '21',
            '22', '23', '24',
            '25', '26', '27',
            '28', '29', '30',
            '31',
        ],
        'time': [
            '00:00', '01:00', '02:00',
            '03:00', '04:00', '05:00',
            '06:00', '07:00', '08:00',
            '09:00', '10:00', '11:00',
            '12:00', '13:00', '14:00',
            '15:00', '16:00', '17:00',
            '18:00', '19:00', '20:00',
            '21:00', '22:00', '23:00',
        ],
        'area': [
            50, -124, 47,
            -122,
        ],
        'format': 'netcdf',
    },
    '2020-2023_JJA_PugetSound_T2m-LSM.nc')

### <br> Using the Daily Statistics Application to get daily means instead of pulling hourly data and calculating it <br> https://cds.climate.copernicus.eu/cdsapp#!/software/app-c3s-daily-era5-statistics?tab=overview <br><br> From this forum post: https://forum.ecmwf.int/t/retrieve-daily-era5-era5-land-data-using-the-cds-api/1150/1 <br><br> After running - note each month is roughly 128MB or about 1.5GB per year! <br> And, it took about 10 min total to retrieve 12 months (12 files)

In [2]:
import cdsapi
import requests
 
# CDS API script to use CDS service to retrieve daily ERA5* variables and iterate over
# all months in the specified years.
 
# Requires:
# 1) the CDS API to be installed and working on your system
# 2) You have agreed to the ERA5 Licence (via the CDS web page)
# 3) Selection of required variable, daily statistic, etc
 
# Output:
# 1) separate netCDF file for chosen daily statistic/variable for each month
 
c = cdsapi.Client()#timeout=300)
 
# Uncomment years as required
 
years =  [
#    '1960',
#    '1961','1962','1963','1964'
 #   '1965',
#    '1966','1967','1968','1969'
#            '1970'
#    '1971','1972','1973','1974','1975'
#    '1976','1977','1978','1979',
#            '1980', 
#    '1981','1982', '1983', '1984',
#            '1985',
#'1986', '1987','1988', '1989', 
#    '1990'#,
#            '1991', '1992', '1993',
#            '1994', '1995', '1996',
#            '1997', '1998', '1999'#,
#            '2000',
#    '2001', '2002',
#            '2003', '2004', '2005',
#            '2006', '2007', '2008',
#            '2009', 
#   '2010', 
#    '2011',
            # '2012', '2013', '2014',
            # '2015', '2016', '2017',
            # '2018', '2019'#, '2020'
#            '2021',
#             '2022'
#    '2023',
    '2024'
]
 
 
# Retrieve all months for a given year.
 
months = ['01', '02', '03',
            '04', '05', '06',
            '07', '08', '09',
            '10', '11', '12']
 
# For valid keywords, see Table 2 of:
# https://datastore.copernicus-climate.eu/documents/app-c3s-daily-era5-statistics/C3S_Application-Documentation_ERA5-daily-statistics-v2.pdf
 
# select your variable; name must be a valid ERA5 CDS API name.
var = "2m_temperature"
 
# Select the required statistic, valid names given in link above
stat = "daily_mean"
 
# Loop over years and months
 
for yr in years:
    for mn in months:
        result = c.service(
        "tool.toolbox.orchestrator.workflow",
        params={
             "realm": "user-apps",
             "project": "app-c3s-daily-era5-statistics",
             "version": "master",
             "kwargs": {
                 "dataset": "reanalysis-era5-single-levels",
                 "product_type": "reanalysis",
                 "variable": var,
                 "statistic": stat,
                 "year": yr,
                 "month": mn,
                 "time_zone": "UTC+00:0",
                 "frequency": "1-hourly",
#
# Users can change the output grid resolution and selected area
#
#                "grid": "1.0/1.0",
#                "area":{"lat": [10, 60], "lon": [65, 140]}
 
                 },
        "workflow_name": "application"
        })
         
# set name of output file for each month (statistic, variable, year, month     
 
        file_name = "../../../Data/ERA5-global/Analysis/" + yr + "/download_" + stat + "_" + var + "_" + yr + "_" + mn + ".nc"
         
        location=result[0]['location']
        res = requests.get(location, stream = True)
        print("Writing data to " + file_name)
        with open(file_name,'wb') as fh:
            for r in res.iter_content(chunk_size = 1024):
                fh.write(r)
        fh.close()

2024-07-04 16:59:42,846 INFO Welcome to the CDS
2024-07-04 16:59:42,847 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-1420d219a6464141876257fe2d93284c
2024-07-04 16:59:43,046 INFO Request is completed


Writing data to ../../../Data/ERA5-global/Analysis/2024/download_daily_mean_2m_temperature_2024_01.nc


2024-07-04 17:00:43,898 INFO Welcome to the CDS
2024-07-04 17:00:43,898 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-9078d696b9c24b2bbe69a6989dd69baa
2024-07-04 17:00:44,122 INFO Request is queued
2024-07-04 17:00:45,307 INFO Request is running
2024-07-04 17:02:39,345 INFO Request is completed


Writing data to ../../../Data/ERA5-global/Analysis/2024/download_daily_mean_2m_temperature_2024_02.nc


2024-07-04 17:03:11,281 INFO Welcome to the CDS
2024-07-04 17:03:11,282 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-ec3da6fe61ef4873aa55c8bc0928d608
2024-07-04 17:03:11,469 INFO Request is queued
2024-07-04 17:03:12,656 INFO Request is running
2024-07-04 17:05:06,686 INFO Request is completed


...

Writing data to ../../../Data/ERA5-global/Analysis/2024/download_daily_mean_2m_temperature_2024_05.nc


2024-07-04 17:11:10,890 INFO Welcome to the CDS
2024-07-04 17:11:10,890 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-7caf95f5b6a64d74a3ae7f6b8333665a


Writing data to ../../../Data/ERA5-global/Analysis/2024/download_daily_mean_2m_temperature_2024_06.nc


2024-07-04 17:11:25,569 INFO Welcome to the CDS
2024-07-04 17:11:25,569 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-cb3bea7ec71e4bc1b90e3ebf29a36ea0
2024-07-04 17:11:25,787 INFO Request is queued
2024-07-04 17:11:26,974 INFO Request is running
2024-07-04 17:11:28,664 INFO Request is failed
2024-07-04 17:11:28,664 ERROR Message: 
2024-07-04 17:11:28,664 ERROR Reason:  Traceback (most recent call last):
  File "/opt/cdstoolbox/cdscompute/cdscompute/cdshandlers/services/handler.py", line 59, in handle_request
    result = cached(context.method, proc, context, context.args, context.kwargs)
  File "/opt/cdstoolbox/cdscompute/cdscompute/caching.py", line 108, in cached
    result = proc(context, *context.args, **context.kwargs)
  File "/opt/cdstoolbox/cdscompute/cdscompute/services.py", line 124, in __call__
    return p(*args, **kwargs)
  File "/opt/cdstoolbox/cdscompute/cdscompute/services.py", line 60, in __ca

Exception: . Traceback (most recent call last):
  File "/opt/cdstoolbox/cdscompute/cdscompute/cdshandlers/services/handler.py", line 59, in handle_request
    result = cached(context.method, proc, context, context.args, context.kwargs)
  File "/opt/cdstoolbox/cdscompute/cdscompute/caching.py", line 108, in cached
    result = proc(context, *context.args, **context.kwargs)
  File "/opt/cdstoolbox/cdscompute/cdscompute/services.py", line 124, in __call__
    return p(*args, **kwargs)
  File "/opt/cdstoolbox/cdscompute/cdscompute/services.py", line 60, in __call__
    return self.proc(context, *args, **kwargs)
  File "/home/cds/cdsservices/services/retrieve.py", line 197, in execute
    remote = context.call_resource(name, request, update_specific_metadata={'app_scope': 'adaptor'})
  File "/opt/cdstoolbox/cdscompute/cdscompute/context.py", line 307, in call_resource
    return c.call_resource(service, *args, **kwargs).value
  File "/opt/cdstoolbox/cdsworkflows/cdsworkflows/future.py", lin

## 

{'r': 327,
 'fh': 168,
 'location': 163,
 'months': 152,
 'open': 152,
 'file_name': 94,
 'result': 88,
 'years': 64,
 'var': 63,
 'c': 56,
 'res': 56,
 'yr': 53,
 'mn': 51}

In [9]:
!ls -al "../../../Data/ERA5-global/Baseline/1989/"

total 2961552
drwxr-xr-x@ 14 tedscott  staff        448 Jun 26 14:39 .
drwxr-xr-x@ 36 tedscott  staff       1152 Jun 26 10:24 ..
-rw-r--r--@  1 tedscott  staff  128778534 Jun 26 14:29 download_daily_mean_2m_temperature_1989_01.nc
-rw-r--r--@  1 tedscott  staff  116319629 Jun 26 14:30 download_daily_mean_2m_temperature_1989_02.nc
-rw-r--r--@  1 tedscott  staff  128778532 Jun 26 14:31 download_daily_mean_2m_temperature_1989_03.nc
-rw-r--r--@  1 tedscott  staff  124625564 Jun 26 14:31 download_daily_mean_2m_temperature_1989_04.nc
-rw-r--r--@  1 tedscott  staff  128778532 Jun 26 14:32 download_daily_mean_2m_temperature_1989_05.nc
-rw-r--r--@  1 tedscott  staff  124625564 Jun 26 14:33 download_daily_mean_2m_temperature_1989_06.nc
-rw-r--r--@  1 tedscott  staff  128778533 Jun 26 14:36 download_daily_mean_2m_temperature_1989_07.nc
-rw-r--r--@  1 tedscott  staff  128778531 Jun 26 14:36 download_daily_mean_2m_temperature_1989_08.nc
-rw-r--r--@  1 tedscott  staff  124625565 Jun 26 14:37 download